# 01: Train a face autoencoder
Trains the main model (CelebA, `configs/faces_64.yaml`) or the secondary one (Cartoon Set, `configs/cartoon_64.yaml`). Logs to Weights & Biases. Re-running resumes from the last checkpoint.

## Setup
Runtime > Change runtime type > **GPU**. Code lives in Google Drive at `MyDrive/carsnfaces`
(upload this folder, or sync it with Google Drive for desktop). Data, checkpoints and outputs also go to Drive,
so a disconnect loses nothing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/carsnfaces'   # code
os.environ['CNF_DATA'] = PROJECT + '/data'     # datasets
os.environ['CNF_RUNS'] = PROJECT + '/runs'     # checkpoints
%cd {PROJECT}
!pip install -q -e . wandb pyyaml kaggle

## Credentials (once per session)
- **W&B:** paste your API key from https://wandb.ai/authorize
- **Kaggle:** put `kaggle.json` (Kaggle > Settings > API > Create New Token) in `MyDrive/carsnfaces/secrets/` (never commit it)
- **Google Cloud (Cartoon Set only):** sign in when prompted

In [ ]:
import wandb
wandb.login()

!mkdir -p ~/.kaggle && cp {PROJECT}/secrets/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## Download data (skips if already on Drive)

In [ ]:
import os
data = os.environ['CNF_DATA']
if not os.path.isdir(f'{data}/raw/celeba'):
    !bash scripts/download_data.sh celeba {data}
if not os.path.isdir(f'{data}/raw/stanford_cars'):
    !bash scripts/download_data.sh cars {data}

In [ ]:
# Only needed for the secondary (cartoon) model
if not os.path.isdir(f'{data}/raw/cartoonset'):
    from google.colab import auth
    auth.authenticate_user()
    !bash scripts/download_data.sh cartoon {data}

## Probe cars
Car fronts shown in W&B during training (`samples/cars_to_faces`). Needs `data/selections/car_fronts.txt` (PLAN.md step 1.3).

In [ ]:
!python scripts/prepare_images.py --input {data}/raw/stanford_cars --list data/selections/car_fronts.txt \
    --out {data}/processed/car_fronts_64 --size 64

## Train
Check the pipeline with the smoke test first, then the real run.

In [ ]:
!python -m carsnfaces.train --config configs/smoke_test.yaml

In [ ]:
CONFIG = 'configs/faces_64.yaml'   # or configs/cartoon_64.yaml
!python -m carsnfaces.train --config {CONFIG}